# Attention Mechanism from Scratch — AI Engineering Interview Prep

Scaled dot-product attention, multi-head attention, positional encoding — in NumPy and PyTorch.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

## 1. Scaled Dot-Product Attention (NumPy)

In [ ]:
def softmax(x, axis=-1):
    e_x = np.exp(x - x.max(axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Args:
        Q: (batch, heads, seq_q, d_k)
        K: (batch, heads, seq_k, d_k)
        V: (batch, heads, seq_k, d_v)
    Returns:
        output: (batch, heads, seq_q, d_v)
        weights: (batch, heads, seq_q, seq_k)
    """
    d_k = Q.shape[-1]
    # Attention scores: QK^T / sqrt(d_k)
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)

    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)  # causal masking

    weights = softmax(scores)    # (batch, heads, seq_q, seq_k)
    output  = weights @ V        # (batch, heads, seq_q, d_v)
    return output, weights

# Example: 1 batch, 1 head, 4 tokens, d_k=8
batch, heads, seq_len, d_k = 1, 1, 4, 8
Q = np.random.randn(batch, heads, seq_len, d_k)
K = np.random.randn(batch, heads, seq_len, d_k)
V = np.random.randn(batch, heads, seq_len, d_k)

output, attn_weights = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {output.shape}")
print(f"Attention weights (should sum to 1 per row):")
print(attn_weights[0, 0].round(4))
print("Row sums:", attn_weights[0, 0].sum(axis=1).round(6))

## 2. Causal (Autoregressive) Masking

In [ ]:
seq_len = 5
# Lower triangular mask — position i can only attend to positions 0..i
causal_mask = np.tril(np.ones((seq_len, seq_len)))
print("Causal mask:")
print(causal_mask)

Q = K = V = np.random.randn(1, 1, seq_len, 8)
_, masked_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
im1 = ax1.imshow(causal_mask, cmap='Blues')
ax1.set_title('Causal Mask')
ax1.set_xlabel('Key position'); ax1.set_ylabel('Query position')

im2 = ax2.imshow(masked_weights[0, 0], cmap='YlOrRd')
ax2.set_title('Masked Attention Weights')
plt.colorbar(im2, ax=ax2)
plt.tight_layout()
plt.show()

## 3. Multi-Head Attention (PyTorch)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def split_heads(self, x):
        # (batch, seq, d_model) → (batch, heads, seq, d_k)
        B, S, _ = x.shape
        return x.view(B, S, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, x, mask=None):
        B, S, _ = x.shape
        Q = self.split_heads(self.W_Q(x))
        K = self.split_heads(self.W_K(x))
        V = self.split_heads(self.W_V(x))

        scores = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        context = weights @ V  # (B, heads, S, d_k)

        # Concatenate heads
        context = context.transpose(1, 2).contiguous().view(B, S, self.d_model)
        return self.W_O(context), weights

d_model, n_heads, seq_len = 64, 4, 10
mha = MultiHeadAttention(d_model, n_heads)
x = torch.randn(2, seq_len, d_model)  # batch=2
out, attn = mha(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Attn shape:   {attn.shape}  (batch, heads, seq, seq)")

## 4. Positional Encoding

In [ ]:
def positional_encoding(max_len, d_model):
    pe = np.zeros((max_len, d_model))
    pos = np.arange(max_len)[:, np.newaxis]          # (max_len, 1)
    i   = np.arange(0, d_model, 2)[np.newaxis, :]   # (1, d_model/2)
    div_term = np.exp(-i * np.log(10000) / d_model)
    pe[:, 0::2] = np.sin(pos * div_term)
    pe[:, 1::2] = np.cos(pos * div_term)
    return pe

pe = positional_encoding(50, 64)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.imshow(pe, aspect='auto', cmap='RdBu')
ax1.set_xlabel('Dimension'); ax1.set_ylabel('Position')
ax1.set_title('Positional Encoding')
plt.colorbar(ax1.images[0], ax=ax1)

for d in [0, 4, 8, 16]:
    ax2.plot(pe[:, d], label=f'd={d}')
ax2.set_xlabel('Position'); ax2.set_ylabel('Value')
ax2.set_title('PE Values by Dimension'); ax2.legend()
plt.tight_layout()
plt.show()

## 5. Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attn   = MultiHeadAttention(d_model, n_heads)
        self.norm1  = nn.LayerNorm(d_model)
        self.norm2  = nn.LayerNorm(d_model)
        self.ff     = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
        )
        self.drop   = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention + residual
        attn_out, _ = self.attn(x, mask)
        x = self.norm1(x + self.drop(attn_out))  # residual connection
        # Feed-forward + residual
        x = self.norm2(x + self.drop(self.ff(x)))
        return x

block = TransformerBlock(d_model=64, n_heads=4, ff_dim=256)
x = torch.randn(2, 10, 64)
out = block(x)
print(f"TransformerBlock: {x.shape} → {out.shape}")
n_params = sum(p.numel() for p in block.parameters())
print(f"Parameters: {n_params:,}")

## Interview Tips

- **Why scale by √d_k?** Dot products grow large with d_k, pushing softmax into saturated regions with tiny gradients.
- **Multi-head attention**: each head learns different aspects (syntax, semantics, coreference).
- **Q, K, V intuition**: Q=what am I looking for, K=what do I have, V=what to return if match.
- **Residual connections**: x + Attention(x) allows gradient flow in deep networks (bypass).
- **LayerNorm**: normalises across feature dimension (not batch) — works well for variable-length sequences.
- **Causal mask**: GPT-style autoregressive generation. Encoder (BERT-style) uses bidirectional attention.